# MCP Agents

This notebook demonstrates how to bind **Model Context Protocol (MCP)** servers to an LLM agent.

MCP decouples tool *discovery* from tool *definition* — tools are served by external processes
and discovered at runtime, so the same agent code works with any MCP-compatible server.

Three binding patterns are covered:

| Pattern | Transport | Best for |
|---|---|---|
| **Anthropic beta API** | Remote HTTP only | Hosted/public servers |
| **Manual MCP bridge** | stdio or HTTP | Full control, local servers |
| **LangChain MCP adapters** | stdio or HTTP | LangGraph integration |

**Sections**
1. Setup
2. MCP Architecture
3. Build an MCP Server with FastMCP
4. Inspect Tools from a Server
5. Binding — Anthropic Beta API
6. Binding — Manual MCP Bridge
7. Binding — LangChain MCP Adapters
8. Summary

---
## 1. Setup & Installation

In [1]:
%pip install -q anthropic mcp deepagents biopython langchain-mcp-adapters langchain-anthropic langgraph python-dotenv
# Filesystem MCP server requires Node.js — brew install node or https://nodejs.org
# ToolUniverse requires uv — brew install uv or https://docs.astral.sh/uv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-experimental 0.3.4 requires langchain-core<0.4.0,>=0.3.28, but you have langchain-core 1.4.0 which is incompatible.
embedchain 0.1.128 requires langchain<0.4.0,>=0.3.1, but you have langchain 1.3.1 which is incompatible.
embedchain 0.1.128 requires langsmith<0.4.0,>=0.3.18, but you have langsmith 0.8.5 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import anthropic
from dotenv import load_dotenv
from pathlib import Path

load_dotenv('../.env')
client = anthropic.Anthropic()
print("Ready.")

Ready.


---
## 2. MCP Architecture

MCP uses a **client–server** model. The host application embeds an MCP client that
connects to one or more MCP servers over a transport.

```
Host application (your agent code)
└── MCP client
    ├── Server A  (stdio)  — local subprocess, e.g. filesystem, PubMed
    ├── Server B  (HTTP)   — remote service, e.g. biocontext.ai, STRING DB
    └── Server C  (stdio)  — local biomedical tools, e.g. ToolUniverse
```

**Transport types**

| Transport | How it works | Use when |
|---|---|---|
| `stdio` | Client spawns a subprocess; communicates over stdin/stdout | Local tools, development |
| Streamable HTTP | Client POSTs to a URL | Hosted services, production |

**What servers expose**

| Primitive | Who decides when to use it | Purpose |
|---|---|---|
| **Tools** | Model (LLM) | Actions, API calls, database queries |
| **Resources** | App (host) | Static context — files, documents |
| **Prompts** | User | Reusable prompt templates |

> This notebook focuses on **tools** — the most common primitive for agentic workflows.

---
## 3. Build an MCP Server with FastMCP

`FastMCP` (included in the `mcp` package) lets you define MCP tools with Python decorators,
similar to FastAPI routes. The server below is adapted from
[Biomni's PubMed MCP example](https://github.com/snap-stanford/Biomni/blob/main/biomni/tool/example_mcp_tools/pubmed_mcp.py).

FastMCP automatically:
- Generates the tool's **JSON schema** from the function signature and Pydantic types
- Uses the **docstring** as the tool description passed to the LLM
- Handles **serialization** of return values

In [3]:
import json
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

pubmed_server = StdioServerParameters(
    command="python",
    args=["server_pubmed.py"],
)

async with stdio_client(pubmed_server) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        search_result = await session.call_tool(
            "search_pubmed",
            {"query": "agentic AI in silico team science biomedical Li", "max_results": 3},
        )

        # FastMCP emits one TextContent item per returned object, each a JSON dict
        articles = [json.loads(item.text) for item in search_result.content]
        print(f"Found {len(articles)} article(s):")
        for a in articles:
            print(f"  PMID {a['pmid']}: {a.get('title', 'N/A')}")

        # Fetch the abstract for the top hit
        pmid = articles[0]["pmid"]
        abstract_result = await session.call_tool("get_abstract", {"pmid": pmid})
        print(f"\nAbstract for PMID {pmid}:")
        print(abstract_result.content[0].text[:600], "...")

Found 1 article(s):
  PMID 41735549: Agentic AI and the rise of in silico team science in biomedical research.

Abstract for PMID 41735549:
1. Nat Biotechnol. 2026 May;44(5):711-725. doi: 10.1038/s41587-026-03035-1. Epub 
2026 Feb 24.

Agentic AI and the rise of in silico team science in biomedical research.

Li B(#)(1), Saini AK(#)(1), Hernandez JG(#)(1), Moore JH(2).

Author information:
(1)Cedars-Sinai Medical Center, Los Angeles, CA, USA.
(2)Cedars-Sinai Medical Center, Los Angeles, CA, USA. jason.moore@csmc.edu.
(#)Contributed equally

Agentic artificial intelligence (AI) systems are emerging as teams of 
intelligent computational experts capable of rivaling human performance in 
labor-intensive tasks, including literature re ...


### Subagent orchestration

Subagents solve the **context bloat problem**: instead of the supervisor accumulating all intermediate tool outputs, it delegates a task and gets back only a final summary. The supervisor picks which subagent to call based on each subagent's `description`; it invokes them via the built-in `task()` tool.

This is distinct from:
- **LangGraph** — where handoffs require explicit graph edges and state routing
- **LangChain `create_react_agent`** — which runs a single flat agent with no context isolation

In [ ]:
from deepagents import create_deep_agent

def get_gene_info(gene_symbol: str) -> dict:
    """Retrieve GC content, chromosome, and length for a gene by its HGNC symbol."""
    db = {
        "BRCA1": {"chromosome": "17q21.31", "length_bp": 81189, "gc_content_pct": 42.1},
        "TP53":  {"chromosome": "17p13.1",  "length_bp": 19149, "gc_content_pct": 50.3},
    }
    return db.get(gene_symbol.upper(), {"error": "Gene not found"})

def interpret_p_value(p_value: float, alpha: float = 0.05) -> dict:
    """Interpret a p-value in the context of hypothesis testing."""
    significant = p_value < alpha
    return {
        "p_value": p_value,
        "alpha": alpha,
        "significant": significant,
        "interpretation": (
            f"p={p_value} < α={alpha}: reject null hypothesis"
            if significant
            else f"p={p_value} ≥ α={alpha}: fail to reject null hypothesis"
        ),
    }

# Each subagent is a dict: name + description (used for routing) + system_prompt + tools
subagents = [
    {
        "name": "gene-specialist",
        "description": "Looks up gene properties (GC content, chromosome, length) by HGNC symbol",
        "system_prompt": (
            "You are a genomics specialist. Use get_gene_info to answer gene questions. "
            "Return only the relevant facts — not raw tool output."
        ),
        "tools": [get_gene_info],
    },
    {
        "name": "stats-specialist",
        "description": "Interprets statistical results such as p-values and significance tests",
        "system_prompt": (
            "You are a biostatistics specialist. Use interpret_p_value to assess results. "
            "Explain implications clearly and concisely."
        ),
        "tools": [interpret_p_value],
    },
]

supervisor = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    system_prompt=(
        "You coordinate genomics and statistics questions. "
        "Delegate gene lookups to gene-specialist and statistical questions to stats-specialist. "
        "Use subagents for specialized tasks to keep your own context clean."
    ),
    subagents=subagents,
)

result = supervisor.invoke({
    "messages": [{
        "role": "user",
        "content": "Compare BRCA1 and TP53 GC content, then tell me if p=0.03 is significant at α=0.05.",
    }]
})
print("Supervisor with subagents — final answer:")
print(result["messages"][-1].content)

### Structured subagent output

Setting `response_format` on a subagent dict forces it to return validated JSON matching a Pydantic schema. The parent receives a `ToolMessage` with JSON content instead of free-form text — the same guarantee as `with_structured_output()` in Section 8, but applied at the subagent boundary so the supervisor can process results programmatically.

In [11]:
from pydantic import BaseModel, Field
from deepagents import create_deep_agent

class GeneReport(BaseModel):
    gene_symbol: str      = Field(description="Gene symbol")
    chromosome: str       = Field(description="Chromosomal location")
    gc_content_pct: float = Field(description="GC content as a percentage")
    length_bp: int        = Field(description="Gene length in base pairs")
    summary: str          = Field(description="One-sentence biological summary")

def get_gene_info(gene_symbol: str) -> dict:
    """Retrieve GC content, chromosome, and length for a gene by its HGNC symbol."""
    db = {
        "BRCA1": {"chromosome": "17q21.31", "length_bp": 81189, "gc_content_pct": 42.1},
        "TP53":  {"chromosome": "17p13.1",  "length_bp": 19149, "gc_content_pct": 50.3},
    }
    return db.get(gene_symbol.upper(), {"error": "Gene not found"})

subagents = [
    {
        "name": "gene-reporter",
        "description": "Looks up a gene and returns a structured JSON report",
        "system_prompt": "Use get_gene_info to look up the gene and return a complete structured report.",
        "tools": [get_gene_info],
        "response_format": GeneReport,   # parent receives GeneReport-shaped JSON, not free text
    }
]

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    subagents=subagents,
)

result = agent.invoke({"messages": [{"role": "user", "content": "Get a structured report for TP53."}]})
print("Structured subagent output (JSON from gene-reporter subagent):")
print(result["messages"][-1].content)

/Users/lib/GitHub/agent-playground/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[05/20/26 17:19:21] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=435794;file:///Users/lib/GitHub/agent-playground/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=249843;file:///Users/lib/GitHub/agent-playground/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[05/20/26 17:21:13] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=293201;file:///Users/lib/GitHub/agent-playground/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=135878;file:///Users/lib/GitHub/agent-playground/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[05/20/26 17:21:19] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=406401;file:///Users/lib/GitHub/agent-playground/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=201142;file:///Users/lib/GitHub/agent-playground/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[05/20/26 17:21:23] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=787884;file:///Users/lib/GitHub/agent-playground/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=81268;file:///Users/lib/GitHub/agent-playground/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

Structured subagent output (JSON from gene-reporter subagent):
Here is the structured report for **TP53**:

| Field | Details |
|---|---|
| **Gene Symbol** | TP53 |
| **Chromosome Location** | 17p13.1 |
| **Gene Length** | 19,149 bp |
| **GC Content** | 50.3% |
| **Summary** | TP53 encodes the tumor protein p53, a critical transcription factor and tumor suppressor that regulates the cell cycle, apoptosis, and genomic stability in response to DNA damage and cellular stress. |

### Key Highlights
- **Role:** Tumor suppressor — often called the *"guardian of the genome"*
- **Function:** Acts as a transcription factor that activates genes involved in DNA repair, cell cycle arrest, and apoptosis when cellular stress or DNA damage is detected.
- **Clinical Significance:** TP53 is the most frequently mutated gene in human cancers, with mutations found in ~50% of all tumor types. Loss-of-function mutations lead to uncontrolled cell proliferation and genomic instability.


---
## 4. Inspect Tools from an MCP Server

The MCP Python SDK's `ClientSession` connects to any MCP server and lists its tools,
regardless of what language the server is written in.

The **filesystem server** (`@modelcontextprotocol/server-filesystem`) is the simplest
server to test with — it exposes local filesystem operations as MCP tools with no
configuration beyond specifying which directory to allow.

> **Prerequisite**: `npx` from Node.js — `brew install node` or https://nodejs.org

In [5]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Filesystem server: allows read/write operations within the specified directory
fs_server = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", str(Path.cwd())],
)

async def list_server_tools(server_params):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.list_tools()
            return result.tools

fs_tools = await list_server_tools(fs_server)
print(f"Filesystem server exposes {len(fs_tools)} tools:\n")
for t in fs_tools:
    print(f"  {t.name}: {(t.description or '')[:70]}")

Filesystem server exposes 14 tools:

  read_file: Read the complete contents of a file as text. DEPRECATED: Use read_tex
  read_text_file: Read the complete contents of a file from the file system as text. Han
  read_media_file: Read an image or audio file. Returns the base64 encoded data and MIME 
  read_multiple_files: Read the contents of multiple files simultaneously. This is more effic
  write_file: Create a new file or completely overwrite an existing file with new co
  edit_file: Make line-based edits to a text file. Each edit replaces exact line se
  create_directory: Create a new directory or ensure a directory exists. Can create multip
  list_directory: Get a detailed listing of all files and directories in a specified pat
  list_directory_with_sizes: Get a detailed listing of all files and directories in a specified pat
  directory_tree: Get a recursive tree view of files and directories as a JSON structure
  move_file: Move or rename files and directories. Can move files be

In [6]:
# Each tool's inputSchema is a JSON Schema — this is passed directly to the LLM
# so it knows how to call the tool correctly
read_tool = next(t for t in fs_tools if t.name == "read_file")

print(f"Tool: {read_tool.name}")
print(f"Description: {read_tool.description}")
print(f"\nInput schema (sent to the LLM):")
print(json.dumps(read_tool.inputSchema, indent=2))

Tool: read_file
Description: Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.

Input schema (sent to the LLM):
{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "type": "object",
  "properties": {
    "path": {
      "type": "string"
    },
    "tail": {
      "description": "If provided, returns only the last N lines of the file",
      "type": "number"
    },
    "head": {
      "description": "If provided, returns only the first N lines of the file",
      "type": "number"
    }
  },
  "required": [
    "path"
  ]
}


---
## 5. Binding — Anthropic Beta API

The simplest binding: pass an `mcp_servers` list to `client.beta.messages.create()`.
Anthropic's API connects to the server, fetches available tools, and manages the
`tool_use` / `tool_result` loop automatically — you never write loop logic.

**Key constraint**: only works with **remote (HTTP) servers**, not local stdio servers.

In [ ]:
# WITHOUT MCP: tools are defined manually as JSON schemas in your code.
# Adding a new tool = update code + redeploy.
manual_tools = [
    {
        "name": "search_knowledge_base",
        "description": "Search a biological knowledge base.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    }
]
print("WITHOUT MCP — tools hardcoded in agent code:")
print(f"  {len(manual_tools)} tool(s) defined manually")
print(f"  Schema, execution logic, and tool list are all coupled to your code")

In [12]:
# WITH Anthropic beta MCP: tools are discovered from the server at request time.
# Swapping servers = different tools, zero code changes.
#
# Public servers to try (no auth required):
#   https://biocontext-kb.fastmcp.app/mcp  — BioContext knowledge base
#   https://mcp.string-db.org/             — STRING protein interaction DB

response = client.beta.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    messages=[{"role": "user", "content": "What tools do you have available? List them briefly."}],
    mcp_servers=[
        {
            "type": "url",
            "url": "https://biocontext-kb.fastmcp.app/mcp",
            "name": "biocontext",
        }
    ],
    extra_headers={"anthropic-beta": "mcp-client-2025-04-04"},
)
print("WITH Anthropic beta MCP (tools from biocontext.ai):")
print(response.content[0].text)

[05/20/26 17:21:33] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true     ]8;id=875198;file:///Users/lib/GitHub/agent-playground/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=78416;file:///Users/lib/GitHub/agent-playground/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

WITH Anthropic beta MCP (tools from biocontext.ai):
Here are the tools I have available, grouped by category:

---

### 🧬 Protein & Gene Information
- **get_uniprot_id_by_protein_symbol** – Convert protein/gene name to UniProt accession ID
- **get_uniprot_protein_info** – Retrieve detailed protein info from UniProt
- **get_alphafold_info_by_protein_symbol** – Fetch AlphaFold structure predictions
- **get_ensembl_id_from_gene_symbol** – Convert gene symbol to Ensembl ID
- **get_protein_domains** – Get protein domain architecture via InterPro
- **get_human_protein_atlas_info** – Expression, localization, and pathology data

---

### 🔬 Proteomics & Structural Biology
- **get_pride_project** – Detailed metadata for a PRIDE proteomics project
- **search_pride_projects** – Search PRIDE database for MS proteomics datasets
- **search_pride_proteins** – Find proteins identified in a PRIDE project

---

### 🕸️ Protein Interactions & Pathways
- **get_string_id** – Map protein names to STRING data

In [ ]:
# Ask a substantive question — the API handles the tool call loop internally
response = client.beta.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    messages=[{"role": "user", "content": "What is the STRING database used for?"}],
    mcp_servers=[{"type": "url", "url": "https://biocontext-kb.fastmcp.app/mcp", "name": "biocontext"}],
    extra_headers={"anthropic-beta": "mcp-client-2025-04-04"},
)
print(response.content[0].text)

---
## 6. Binding — Manual MCP Bridge

When you need to use **stdio servers** (local subprocesses) or want full visibility
into each tool call, bridge MCP to the Anthropic API manually:

1. Connect to the server with `ClientSession`
2. Convert MCP tool schemas → Anthropic tool format
3. Route tool calls back through `session.call_tool()`
4. Run the standard `tool_use` / `tool_result` message loop

This works with **any** MCP server regardless of transport.

In [ ]:
def mcp_to_anthropic_tool(mcp_tool) -> dict:
    """Convert an MCP tool definition to Anthropic's tool format."""
    return {
        "name": mcp_tool.name,
        "description": mcp_tool.description or "",
        "input_schema": mcp_tool.inputSchema,
    }


def extract_text(content_list) -> str:
    """Pull plain text out of an MCP tool result content list."""
    parts = []
    for item in content_list:
        if hasattr(item, "text"):
            parts.append(item.text)
        else:
            parts.append(str(item))
    return "\n".join(parts)


async def run_agent(query: str, server_params: StdioServerParameters, verbose: bool = True) -> str:
    """Run a Claude agent that discovers and calls tools from an MCP server."""
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Step 1: discover tools from the server
            mcp_tools = (await session.list_tools()).tools
            anthropic_tools = [mcp_to_anthropic_tool(t) for t in mcp_tools]

            if verbose:
                print(f"Loaded {len(anthropic_tools)} tools from MCP server")

            messages = [{"role": "user", "content": query}]

            # Step 2: agent loop
            while True:
                response = client.messages.create(
                    model="claude-sonnet-4-6",
                    max_tokens=1024,
                    tools=anthropic_tools,
                    messages=messages,
                )

                if response.stop_reason == "end_turn":
                    return next(
                        (b.text for b in response.content if hasattr(b, "text")), ""
                    )

                # Step 3: execute tool calls and return results
                messages.append({"role": "assistant", "content": response.content})
                tool_results = []

                for block in response.content:
                    if block.type == "tool_use":
                        if verbose:
                            print(f"  → {block.name}({list(block.input.keys())})")
                        result = await session.call_tool(block.name, block.input)
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": extract_text(result.content),
                        })

                messages.append({"role": "user", "content": tool_results})

print("Helper functions defined.")

In [ ]:
# Example: agent using the filesystem server to explore the current directory
answer = await run_agent(
    "List the files in this directory. For each .py or .ipynb file, give one sentence describing what it likely does.",
    fs_server,
)
print("\nAgent answer:")
print(answer)

In [ ]:
# Same agent code, different server = different tools, different domain
# Swap to the PubMed server by changing one line
pubmed_server = StdioServerParameters(
    command="python",
    args=["server_pubmed.py"],
)

answer = await run_agent(
    "Find 3 recent papers about CRISPR base editing and summarize the key themes.",
    pubmed_server,
)
print("\nPubMed agent answer:")
print(answer)

---
## 7. Binding — LangChain MCP Adapters

`langchain-mcp-adapters` converts MCP tools into LangChain `BaseTool` objects so they
work with any LangChain-compatible agent, including LangGraph's `create_react_agent`.

`MultiServerMCPClient` manages multiple server connections under one context manager.
The `mcp_config` dict mirrors the format used in
[BaseAgent's `mcp_config.yaml`](https://github.com/BinglanLi/BaseAgent/blob/main/examples/mcp_config.yaml).

In [ ]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_anthropic import ChatAnthropic
from langgraph.prebuilt import create_react_agent

# MCP server config — same structure as BaseAgent's mcp_config.yaml.
# server_pubmed.py (written in Section 3) is used as a stdio server here.
mcp_config = {
    "pubmed": {
        "command": "python",
        "args": ["server_pubmed.py"],
        "transport": "stdio",
        "env": {**os.environ, "NCBI_EMAIL": os.getenv("NCBI_EMAIL", "your-email@example.com")},
    },
    # Uncomment to add more servers alongside pubmed:
    # "filesystem": {
    #     "command": "npx",
    #     "args": ["-y", "@modelcontextprotocol/server-filesystem", str(Path.cwd())],
    #     "transport": "stdio",
    # },
    # "tooluniverse": {
    #     "command": "uvx",
    #     "args": ["--refresh", "tooluniverse"],
    #     "transport": "stdio",
    # },
}

async with MultiServerMCPClient(mcp_config) as mcp_client:
    lc_tools = mcp_client.get_tools()
    print(f"Loaded {len(lc_tools)} LangChain tools from MCP:")
    for t in lc_tools:
        print(f"  {t.name}: {(t.description or '')[:80]}")

In [ ]:
# LangGraph ReAct agent using the PubMed MCP server.
# The agent calls search_pubmed and get_abstract from server_pubmed.py
# to look up the paper and retrieve its details.
async with MultiServerMCPClient(mcp_config) as mcp_client:
    lc_tools = mcp_client.get_tools()

    agent = create_react_agent(
        ChatAnthropic(model="claude-sonnet-4-6"),
        lc_tools,
    )

    result = await agent.ainvoke({
        "messages": [{
            "role": "user",
            "content": (
                "Search PubMed for the paper titled "
                "'Agentic AI and the rise of in silico team science in biomedical research' "
                "by Binglan Li, published in Nature Biotechnology in 2026. "
                "Return the PMID, full author list, and a two-sentence summary of what the paper is about."
            ),
        }]
    })

print("LangGraph ReAct agent — PubMed search result:")
print(result["messages"][-1].content)

In [ ]:
# Mixing MCP tools with hand-written LangChain tools.
# The agent can call both PubMed MCP tools and the custom gene lookup in one turn.
from langchain_core.tools import tool

@tool
def get_gene_gc_content(gene_symbol: str) -> dict:
    """Return GC content and chromosome location for a gene symbol."""
    db = {
        "BRCA1": {"chromosome": "17q21.31", "gc_content_pct": 42.1},
        "TP53":  {"chromosome": "17p13.1",  "gc_content_pct": 50.3},
    }
    return db.get(gene_symbol.upper(), {"error": "Gene not found"})

async with MultiServerMCPClient(mcp_config) as mcp_client:
    all_tools = mcp_client.get_tools() + [get_gene_gc_content]
    print(f"Total tools: {len(all_tools)}  "
          f"(MCP: {len(mcp_client.get_tools())}, custom: 1)")

    agent = create_react_agent(
        ChatAnthropic(model="claude-sonnet-4-6"),
        all_tools,
    )

    result = await agent.ainvoke({
        "messages": [{
            "role": "user",
            "content": (
                "Search PubMed for recent papers on BRCA1 gene editing (max 3 results). "
                "Also look up the GC content and chromosome of BRCA1. "
                "Summarize both in a few sentences."
            ),
        }]
    })

print("\nAgent answer (PubMed MCP + custom gene tool):")
print(result["messages"][-1].content)

---
## 8. Summary

| Binding pattern | Transport | Loop handled by | Visibility |
|---|---|---|---|
| **Anthropic beta API** | Remote HTTP only | Anthropic API | Hidden |
| **Manual MCP bridge** | stdio or HTTP | Your code | Full |
| **LangChain MCP adapters** | stdio or HTTP | LangGraph | Step-level |

**When to use which**
- **Anthropic beta API** — server is hosted publicly (biocontext.ai, STRING DB); you want minimal code.
- **Manual bridge** — local stdio server; need to inspect or post-process every tool call.
- **LangChain adapters** — building a LangGraph workflow; mixing MCP tools with other LangChain tools.

**MCP servers used in this notebook**

| Server | Transport | Command / URL |
|---|---|---|
| Filesystem | stdio | `npx -y @modelcontextprotocol/server-filesystem ./` |
| PubMed (custom) | stdio | `python server_pubmed.py` |
| BioContext | HTTP | `https://biocontext-kb.fastmcp.app/mcp` |
| ToolUniverse | stdio | `uvx --refresh tooluniverse` |
| STRING DB | HTTP | `https://mcp.string-db.org/` |